# Notebook 14G — Final Fannie compatibility decisions
Implement candidate harmonization on the reviewed diagnostic sample. Propose a conservative companion feature contract and explicitly exclude unresolved predictors from its primary version. No training, scoring, target relabeling or changes to Freddie artifacts. The contract remains conditional on release and outcome approval.

References: [Fannie July 2024 glossary](https://capitalmarkets.fanniemae.com/sites/capmrkt/files/2024-07/crt-file-layout-and-glossary.pdf), [Freddie guide](https://www.freddiemac.com/fmac-resources/research/pdf/user_guide.pdf), [Freddie disclosure changes](https://www.freddiemac.com/fmac-resources/research/pdf/disclosure-changes-summary.pdf), [Fannie 2026 disclosure changes](https://capitalmarkets.fanniemae.com/credit-risk-transfer/single-family-credit-risk-transfer/enhance-credit-risk-transfer-disclosures), [Fannie purpose codes](https://capitalmarkets.fanniemae.com/credit-risk-transfer/single-family-credit-risk-transfer/fannie-mae-enhance-single-family-loan-performance-credit-dataset-january-2015-release). The file's reporting cutoff is not proof of its release version. The 2026 changes make that distinction important.

## 1. Verify reference artifacts and prior outputs

In [1]:
from pathlib import Path
import ast, hashlib, json, re, zipfile
from collections import Counter
from functools import lru_cache
from datetime import datetime, timezone
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
SOURCE = ROOT/'data/raw/Fannie_Mae/2018Q1/2018Q1.csv'
BASE = ROOT/'data/processed/Freddie_Mac/model_baselines_2015_2017'
FEATURE_SOURCE = ROOT/'src/freddie_features.py'
manifest = json.loads((BASE/'model_manifest.json').read_text(encoding='utf-8'))
contract_file = BASE/'freddie_feature_contract.json'
contract = json.loads(contract_file.read_text(encoding='utf-8'))
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
assert sha(contract_file) == manifest['feature_contract_sha256']
assert manifest['horizon_months'] == 24
constants = {t.id: ast.literal_eval(node.value) for node in ast.parse(FEATURE_SOURCE.read_text(encoding='utf-8')).body if isinstance(node,ast.Assign) for t in node.targets if isinstance(t,ast.Name) and t.id in {'NUMERIC_FEATURES','CATEGORICAL_FEATURES'}}
NUM = constants['NUMERIC_FEATURES']; CAT = constants['CATEGORICAL_FEATURES']
assert NUM+CAT == manifest['predictors'] and len(NUM+CAT)==29
assert set(NUM)==set(contract['numeric_medians']) and set(CAT)==set(contract['category_levels'])
assert contract['training_years']==[2015,2016,2017]
print('Verified frozen contract:', len(NUM), 'numeric and', len(CAT), 'categorical predictors')

Verified frozen contract: 15 numeric and 14 categorical predictors


In [2]:
PROCESSED = ROOT/'data/processed/Fannie_Mae'
# Optional explicit paths to the exact reviewed bundles.
OVERRIDES = {}
def locate(name):
    if name in OVERRIDES:
        return Path(OVERRIDES[name])
    matches = sorted(PROCESSED.rglob(name))
    if not matches:
        raise FileNotFoundError(f'Cannot locate {name}; set OVERRIDES.')
    if len({sha(p) for p in matches}) > 1:
        raise ValueError(f'Multiple different bundles: {name}. Set OVERRIDES.')
    return matches[-1]
e_path = locate('notebook_14e_predictor_compatibility_review.zip')
d_path = locate('notebook_14c_stage3de_review.zip')
with zipfile.ZipFile(e_path) as z:
    e_meta = json.loads(z.read('metadata.json'))
    mapping = pd.read_csv(z.open('predictor_mapping.csv'))
    prior_profile = pd.read_csv(z.open('source_field_profile.csv'))
with zipfile.ZipFile(d_path) as z:
    d_meta = json.loads(z.read('stage3de_metadata.json'))
    target = pd.read_csv(z.open('target_summary.csv'))
    cohorts = pd.read_csv(z.open('cohort_target_summary.csv'))
    checks = pd.read_csv(z.open('reconciliation_checks.csv'))
for m in [e_meta,d_meta]:
    assert (m['sample_loans'],m['sample_rows'],m['physical_rows']) == (4749,245552,23563399)
    assert m['file_bytes'] == SOURCE.stat().st_size
    assert not m['models_scored']
assert e_meta['frozen_contract_sha256'] == sha(contract_file)
assert e_meta['feature_source_sha256'] == sha(FEATURE_SOURCE)
assert d_meta['freddie_source_sha256'] == sha(ROOT/'src/freddie_performance.py')
assert mapping.predictor.tolist() == NUM+CAT
assert checks.passed.astype(str).str.lower().eq('true').all()
assert target.set_index('policy').eligible.to_dict()=={'primary':4491,'strict':4488}
assert target.composite.eq(33).all()
print('Reviewed evidence reconciled.'); display(target); display(cohorts)

Reviewed evidence reconciled.


,policy,sample_loans,eligible,ineligible,serious,adverse,both,composite,event_rate
0,primary,4749,4491,258,33,3,3,33,0.007348
1,strict,4749,4488,261,33,3,3,33,0.007353


,policy,origination_quarter,loans,eligible,events,event_rate
0,primary,2017Q1,1,0,0,NaN
1,primary,2017Q2,2,0,0,NaN
2,primary,2017Q3,18,0,0,NaN
3,primary,2017Q4,2068,1838,15,0.008161
4,primary,2018Q1,2655,2653,18,0.006785
5,primary,NaN,5,0,0,NaN
6,strict,2017Q1,1,0,0,NaN
7,strict,2017Q2,2,0,0,NaN
8,strict,2017Q3,18,0,0,NaN
9,strict,2017Q4,2068,1837,15,0.008165


In [3]:
f_path = locate('notebook_14f_compatibility_synthesis_review.zip')
with zipfile.ZipFile(f_path) as z:
    f_meta = json.loads(z.read('metadata.json'))
    previous = pd.read_csv(z.open('predictor_decisions.csv'))
    previous_mi = pd.read_csv(z.open('mi_type_crosscheck.csv'))
assert f_meta['prior_14e_zip_sha256'] == sha(e_path)
assert f_meta['prior_14d_zip_sha256'] == sha(d_path)
assert f_meta['frozen_contract_sha256'] == sha(contract_file)
assert previous.predictor.tolist() == NUM + CAT
assert not f_meta['models_scored']
print('14F evidence reconciled.')

14F evidence reconciled.


## 2. Recreate the diagnostic loan sample
Reuse the same scan and earliest reporting snapshot; never backfill from later reporting records.

In [4]:
SIZE = SOURCE.stat().st_size
WIDTH = 113
assert SIZE == 6928164686, 'Source differs from reviewed file.'
POSITIONS = {'loan_id':2,'report':3,'channel':4,'rate':8,'upb':10,'term':13,'orig':14,'first_payment':15,'ltv':20,'cltv':21,'borrowers':22,'dti':23,'borrower_fico':24,'coborrower_fico':25,'fthb':26,'purpose':27,'property_type':28,'units':29,'occupancy':30,'state':31,'mi':34,'amortization':35,'interest_only':37,'mi_type':73,'high_balance':87}
SAMPLE_MODULUS = 100
MAX_SAMPLE_ROWS = 1_000_000
SOURCE_SIGNATURE = (SOURCE.stat().st_size, SOURCE.stat().st_mtime_ns)
@lru_cache(maxsize=100_000)
def selected(loan_id):
    return int.from_bytes(hashlib.blake2b(loan_id,digest_size=8,person=b'Fannie14v1').digest(),'big') % SAMPLE_MODULUS == 0
def scan_source(path, expected_width, progress_every=2_000_000):
    counts = Counter()
    sampled = []
    widths_seen = Counter()
    id_pattern = re.compile(rb'[A-Za-z0-9]{1,12}')
    indexes = [p-1 for p in POSITIONS.values()]
    with path.open('rb') as stream:
        for line_number, raw in enumerate(stream, 1):
            parts = raw.rstrip(b'\r\n').split(b'|')
            if line_number == 1:
                parts[0] = parts[0].removeprefix(b'\xef\xbb\xbf')
            counts['physical_rows'] += 1
            widths_seen[len(parts)] += 1
            if len(parts) != expected_width:
                counts['wrong_width'] += 1
                continue
            loan_id = parts[1].strip()
            if not id_pattern.fullmatch(loan_id):
                counts['invalid_loan_id'] += 1
                continue
            counts['valid_id_rows'] += 1
            if selected(loan_id):
                sampled.append(tuple(parts[i].strip().decode('ascii') for i in indexes))
                if len(sampled) > MAX_SAMPLE_ROWS:
                    raise RuntimeError('Sample exceeds memory guard; increase SAMPLE_MODULUS and rerun the entire scan.')
            if progress_every and line_number % progress_every == 0:
                print(f'{line_number:,} rows scanned; {len(sampled):,} sampled rows')
    return pd.DataFrame(sampled, columns=list(POSITIONS)), counts, widths_seen

history, scan_counts, full_widths = scan_source(SOURCE, WIDTH)
assert SOURCE_SIGNATURE == (SOURCE.stat().st_size, SOURCE.stat().st_mtime_ns), 'Source changed during scan.'
print(dict(scan_counts), dict(full_widths))
assert not scan_counts['wrong_width'] and not scan_counts['invalid_loan_id'], 'Malformed records found; review before continuing.'
assert not history.empty, 'No sampled rows'
print('Sample rows:', len(history), '| Sample distinct loans:', history.loan_id.nunique())
assert scan_counts['physical_rows'] == 23563399
assert len(history) == 245552 and history.loan_id.nunique() == 4749, 'Sample differs from reviewed Stage 2.'

2,000,000 rows scanned; 22,660 sampled rows
4,000,000 rows scanned; 43,404 sampled rows
6,000,000 rows scanned; 63,865 sampled rows
8,000,000 rows scanned; 83,683 sampled rows
10,000,000 rows scanned; 103,966 sampled rows
12,000,000 rows scanned; 123,590 sampled rows
14,000,000 rows scanned; 145,624 sampled rows
16,000,000 rows scanned; 165,802 sampled rows
18,000,000 rows scanned; 186,285 sampled rows
20,000,000 rows scanned; 206,376 sampled rows
22,000,000 rows scanned; 229,575 sampled rows
{'physical_rows': 23563399, 'valid_id_rows': 23563399} {113: 23563399}
Sample rows: 245552 | Sample distinct loans: 4749


In [5]:
assert history.report.str.fullmatch(r'(0[1-9]|1[0-2])(19|20)\d{2}').all()
history['report_idx'] = history.report.str[2:].astype(int)*12 + history.report.str[:2].astype(int)
assert not history.duplicated(['loan_id','report_idx']).any()
first = history.sort_values('report_idx').drop_duplicates('loan_id').copy()
fields = [f for f in POSITIONS if f not in {'loan_id','report'}]
profiles=[]; categories=[]
for field in fields:
    v=first[field]; present=v.ne(''); numeric=pd.to_numeric(v.where(present),errors='coerce')
    distinct=history.groupby('loan_id')[field].nunique(dropna=False)
    nonblank_distinct=history[field].where(history[field].ne('')).groupby(history.loan_id).nunique()
    profiles.append(dict(field=field,physical_position=POSITIONS[field],loans=len(first),blank=int((~present).sum()),nonblank=int(present.sum()),numeric_parseable=int(numeric.notna().sum()),numeric_min=numeric.min(),numeric_max=numeric.max(),loans_with_any_change=int(distinct.gt(1).sum()),loans_with_nonblank_conflict=int(nonblank_distinct.gt(1).sum())))
    if field in ['channel','term','borrowers','fthb','purpose','property_type','units','occupancy','state','amortization','interest_only','mi_type','high_balance']:
        for value,count in v.value_counts().items(): categories.append(dict(field=field,value=value or '[blank]',loans=int(count)))
profile=pd.DataFrame(profiles); category_counts=pd.DataFrame(categories)
display(profile)

,field,physical_position,loans,blank,nonblank,numeric_parseable,numeric_min,numeric_max,loans_with_any_change,loans_with_nonblank_conflict
0,channel,4,4749,0,4749,0,NaN,NaN,1,1
1,rate,8,4749,0,4749,4749,2.75,6.125,0,0
2,upb,10,4749,0,4749,4749,24000.00,1223000.000,0,0
3,term,13,4749,0,4749,4749,96.00,360.000,0,0
4,orig,14,4749,0,4749,4749,12017.00,122017.000,4,4
5,first_payment,15,4749,0,4749,4749,12018.00,122017.000,1,1
6,ltv,20,4749,0,4749,4749,9.00,97.000,0,0
7,cltv,21,4749,0,4749,4749,9.00,105.000,0,0
8,borrowers,22,4749,0,4749,4749,1.00,4.000,0,0
9,dti,23,4749,1,4748,4748,1.00,50.000,1,0


## 3. Implement numerical mappings
Use MI type to distinguish no insurance from unavailable coverage. Freddie uses MI > 0 for the insurance flag, including a false flag when coverage is missing; preserve this behavior for comparability. Treat FICO minimum as a proxy only, and compare with borrower-only scores without using outcomes. Do not select a mapping based on model performance.

In [6]:
import numpy as np

def valid_number(s, low, high):
    v = pd.to_numeric(s, errors='coerce')
    return v.where(v.between(low, high))

def harmonize_mi(percent, kind):
    v = valid_number(percent, 1, 55)
    no_mi = percent.eq('') & kind.eq('')
    insured = kind.isin(['1','2','3'])
    inconsistent = ~(no_mi | (insured & (percent.eq('') | v.notna())))
    mapped = v.where(insured).mask(no_mi, 0.0)
    return mapped, inconsistent

mi, mi_bad = harmonize_mi(first.mi, first.mi_type)
# Synthetic checks for no insurance, valid coverage, unknown coverage, and conflict.
test_mi, test_bad = harmonize_mi(pd.Series(['','25','','25','999']), pd.Series(['','1','2','','1']))
assert test_mi.iloc[:2].tolist() == [0.,25.]
assert test_mi.iloc[2:].isna().all()
assert test_bad.tolist() == [False,False,False,True,True]
primary_score = valid_number(first.borrower_fico,300,850)
co_score = valid_number(first.coborrower_fico,300,850)
min_score = pd.concat([primary_score,co_score],axis=1).min(axis=1)
score_summary = pd.DataFrame([{'mapping':name,'available':int(s.notna().sum()),'missing':int(s.isna().sum()),'mean':s.mean(),'median':s.median()} for name,s in [('primary_borrower',primary_score),('minimum_disclosed',min_score)]])
score_summary['different_on_common_nonmissing'] = int((primary_score.notna() & min_score.notna() & primary_score.ne(min_score)).sum())
mi_summary = pd.DataFrame([{'loans':len(first),'no_insurance':int(mi.eq(0).sum()),'positive_coverage':int(mi.gt(0).sum()),'missing_coverage':int(mi.isna().sum()),'inconsistent':int(mi_bad.sum())}])
display(mi_summary); display(score_summary)
assert not mi_bad.any(), 'Review inconsistent MI records before continuing.'
assert int(mi.eq(0).sum()) == 3388 and int(mi.gt(0).sum()) == 1361


,loans,no_insurance,positive_coverage,missing_coverage,inconsistent
0,4749,3388,1361,0,0


,mapping,available,missing,mean,median,different_on_common_nonmissing
0,primary_borrower,4745,4,747.547313,757.0,1006
1,minimum_disclosed,4745,4,741.797260,750.0,1006


## 4. Construct candidate common predictors
Use the existing 18 candidates plus three MI predictors. Add a coarser purchase/refinance feature as a candidate replacement for loan purpose, conditional on applying the same recode to Freddie. Keep FICO, FTHB, high-balance and product/program indicators out of the conservative contract unless their equivalence is established. Amortization is retained for auditing but may be constant in matched populations. Do not use outcome information or fit preprocessing here. Numeric validity bounds below are diagnostic guardrails, not a certified release-specific cleaning contract.

In [7]:
x = pd.DataFrame(index=first.index)
dti = valid_number(first.dti,1,998)
ltv = valid_number(first.ltv,1,998)
cltv = valid_number(first.cltv,1,998)
upb = pd.to_numeric(first.upb,errors='coerce')
upb = upb.where(upb.gt(0))
x['ORIGINAL_DTI'] = dti
x['ORIGINAL_LTV'] = ltv
x['ORIGINAL_INTEREST_RATE'] = valid_number(first.rate,0.01,100)
x['LOG_ORIGINAL_UPB'] = np.log1p(upb)
x['CLTV_MINUS_LTV'] = (cltv-ltv).clip(lower=0)
x['HAS_SUBORDINATE_FINANCING'] = cltv.gt(ltv).astype('int8')
x['HIGH_LTV_GT_80'] = ltv.gt(80).astype('int8')
for name,s in [('ORIGINAL_DTI',dti),('ORIGINAL_LTV',ltv),('CLTV',cltv)]:
    x[name+'_MISSING'] = s.isna().astype('int8')
x['MI_PERCENT'] = mi
x['HAS_MORTGAGE_INSURANCE'] = mi.gt(0).astype('int8')
x['MI_PERCENT_MISSING'] = mi.isna().astype('int8')
for name,field,allowed in [('NUMBER_OF_UNITS','units',{'1','2','3','4'}),('OCCUPANCY_STATUS','occupancy',{'P','S','I'}),('CHANNEL','channel',{'B','C','R'}),('AMORTIZATION_TYPE','amortization',{'FRM','ARM'}),('PROPERTY_TYPE','property_type',{'SF','PU','CO','CP','MH'})]:
    x[name] = first[field].where(first[field].isin(allowed))
x['PROPERTY_STATE'] = first.state.where(first.state.str.fullmatch('[A-Z]{2}'))
x['ORIGINAL_LOAN_TERM'] = first.term.where(first.term.str.fullmatch('[0-9]+'))
b = pd.to_numeric(first.borrowers,errors='coerce')
x['NUMBER_OF_BORROWERS_GROUP'] = pd.Series(pd.NA,index=first.index,dtype='string').mask(b.eq(1),'1').mask(b.between(2,10),'2+')
x['PURCHASE_REFINANCE_GROUP'] = first.purpose.map({'P':'PURCHASE','C':'REFINANCE','R':'REFINANCE'})
# Apply equivalent P -> PURCHASE, C/N -> REFINANCE on Freddie in the later dataset notebook, only after verifying Freddie purpose codes.
common = [p for p in NUM+CAT if p in x.columns]
assert len(common)==21 and len(x)==4749
candidate = common + ['PURCHASE_REFINANCE_GROUP']
assert list(x[candidate].columns)==candidate
feature_profile = pd.DataFrame([{'predictor':p,'loans':len(x),'missing':int(x[p].isna().sum()),'distinct_nonmissing':int(x[p].nunique())} for p in candidate])
display(feature_profile)


,predictor,loans,missing,distinct_nonmissing
0,MI_PERCENT,4749,0,8
1,ORIGINAL_DTI,4749,1,47
2,ORIGINAL_LTV,4749,0,89
3,ORIGINAL_INTEREST_RATE,4749,0,91
4,LOG_ORIGINAL_UPB,4749,0,525
5,CLTV_MINUS_LTV,4749,0,40
6,HAS_MORTGAGE_INSURANCE,4749,0,2
7,HAS_SUBORDINATE_FINANCING,4749,0,2
8,HIGH_LTV_GT_80,4749,0,2
9,MI_PERCENT_MISSING,4749,0,1


## 5. Record a decision for every original predictor
A conservative conditional companion specification is preferred over direct scoring. FICO proxy and high-balance mappings remain diagnostic alternatives, not approved replacements. No final model eligibility is asserted.

In [8]:
reasons = {
'FICO_SCORE':'Minimum disclosed representative score is a proxy; exact aggregation and release unresolved.',
'FICO_SCORE_MISSING':'Depends on unresolved FICO mapping.',
'FIRST_TIME_HOMEBUYER':'Non-purchase applicability and unknown coding differ.',
'LOAN_PURPOSE':'Replace with coarser purchase/refinance candidate, not literal category equivalence.',
'SUPER_CONFORMING_FLAG':'High-balance analogue observed but exact limits/applicability not established.',
'SPECIAL_ELIGIBILITY_PROGRAM':'No established cross-agency equivalent.',
'HARP_INDICATOR':'Do not invent structural constant without release/population proof.',
'INTEREST_ONLY_INDICATOR':'Observed N does not establish equivalence for all releases.'}
rows=[]
for p in NUM+CAT:
    if p in common:
        action='retain_conditional'
        reason='Use comparable source definition and original transform; release approval required.'
        if p in ['MI_PERCENT','HAS_MORTGAGE_INSURANCE','MI_PERCENT_MISSING']:
            reason='MI type and percentage jointly distinguish no MI from unknown coverage.'
    else:
        action='exclude_from_primary_candidate'
        reason=reasons[p]
    rows.append({'predictor':p,'decision':action,'reason':reason,'final_approved':False})
decisions=pd.DataFrame(rows)
assert len(decisions)==29 and decisions.predictor.is_unique
# Original categorical contract comparison is a diagnostic only, not the companion preprocessor.
overlap=[]
for p in CAT:
    if p in x:
        v=x[p].astype('string'); allowed=contract['category_levels'][p]
        overlap.append({'predictor':p,'missing':int(v.isna().sum()),'outside_original_frozen_levels':int((v.notna() & ~v.isin(allowed)).sum())})
category_overlap=pd.DataFrame(overlap)
display(decisions); display(category_overlap)


,predictor,decision,reason,final_approved
0,FICO_SCORE,exclude_from_primary_candidate,Minimum disclosed representative score is a pr...,False
1,MI_PERCENT,retain_conditional,MI type and percentage jointly distinguish no ...,False
2,ORIGINAL_DTI,retain_conditional,Use comparable source definition and original ...,False
3,ORIGINAL_LTV,retain_conditional,Use comparable source definition and original ...,False
4,ORIGINAL_INTEREST_RATE,retain_conditional,Use comparable source definition and original ...,False
5,LOG_ORIGINAL_UPB,retain_conditional,Use comparable source definition and original ...,False
6,CLTV_MINUS_LTV,retain_conditional,Use comparable source definition and original ...,False
7,HAS_MORTGAGE_INSURANCE,retain_conditional,MI type and percentage jointly distinguish no ...,False
8,HAS_SUBORDINATE_FINANCING,retain_conditional,Use comparable source definition and original ...,False
9,HIGH_LTV_GT_80,retain_conditional,Use comparable source definition and original ...,False


,predictor,missing,outside_original_frozen_levels
0,NUMBER_OF_UNITS,0,0
1,OCCUPANCY_STATUS,0,0
2,CHANNEL,0,0
3,AMORTIZATION_TYPE,0,0
4,PROPERTY_STATE,0,7
5,PROPERTY_TYPE,0,0
6,ORIGINAL_LOAN_TERM,0,23
7,NUMBER_OF_BORROWERS_GROUP,0,0


## 6. Target and population contract
Preserve the existing first-payment month 0–23 composite as a working mapped outcome. Codes 02/03/09/15 have imperfect semantic equivalence. For the serious numeric delinquency sensitivity, reconstruct the numeric-only outcome in both datasets and train separate development-only models if comparing predictions for that outcome; Freddie RA must not be silently included. Use primary and strict eligibility separately. No final labels are created here.

Pilot evaluation: all eligible actual 2018 Q1 originations in the available acquisition file, not a complete origination cohort. Report acquisition coverage limitations. Match Freddie evaluation population restrictions before comparison; avoid selecting based on Fannie outcomes. For a temporal study, obtain additional acquisition files and assess coverage. Freeze the feature contract before any scoring.

In [9]:
RELEASE_EVIDENCE = ''  # Add download date/version and matching glossary reference when available.
gates = pd.DataFrame([
('prior_evidence_reconciled','passed'),('candidate_feature_implementation','passed'),('release_provenance_and_position_semantics','pending'),('cross_agency_target_semantics_and_numeric_only_sensitivity','pending'),('matched_population_and_file_coverage','pending'),('common_contract_review','pending'),('model_scoring','blocked')],columns=['gate','status'])
proposal = dict(strategy='companion_models',original_predictors_retained=common,additional_candidate=['PURCHASE_REFINANCE_GROUP'],excluded_original=[p for p in NUM+CAT if p not in common],fico_sensitivity='minimum disclosed vs primary borrower; no scoring authorized',training_years=[2015,2016,2017],pilot='eligible actual 2018Q1 originations in available acquisition file only',release_evidence=RELEASE_EVIDENCE,final_approved=False,models_scored=False)
display(gates)


,gate,status
0,prior_evidence_reconciled,passed
1,candidate_feature_implementation,passed
2,release_provenance_and_position_semantics,pending
3,cross_agency_target_semantics_and_numeric_only...,pending
4,matched_population_and_file_coverage,pending
5,common_contract_review,pending
6,model_scoring,blocked


## 7. Save aggregate evidence
No loan IDs, loan-level predictors, labels, predictions or models are exported.

In [10]:
out=PROCESSED/'notebook_14g_final_compatibility_decisions'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
out.mkdir(parents=True,exist_ok=False)
for name,table in {'predictor_decisions.csv':decisions,'candidate_feature_profile.csv':feature_profile,'mi_harmonization.csv':mi_summary,'fico_mapping_sensitivity.csv':score_summary,'category_overlap.csv':category_overlap,'review_gates.csv':gates}.items():
    table.to_csv(out/name,index=False)
proposal.update(stage='14G',sample_loans=len(x),sample_rows=len(history),prior_14f_zip_sha256=sha(f_path),frozen_contract_sha256=sha(contract_file),release_verified=False)
(out/'candidate_contract.json').write_text(json.dumps(proposal,indent=2),encoding='utf-8')
review=out/'notebook_14g_final_compatibility_review.zip'
with zipfile.ZipFile(review,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(out.iterdir()):
        if f!=review: z.write(f,f.name)
print('Review ZIP:',review)


Review ZIP: e:\capstone_project\mortgage-model-reliability\data\processed\Fannie_Mae\notebook_14g_final_compatibility_decisions\20261124T161748429287Z\notebook_14g_final_compatibility_review.zip
